# TopDrive AI — Phase 1 Training (InceptionTime Ensemble)

Trains the InceptionTime ensemble classifier on the synthetic dataset
generated by `01_generate_synthetic_data.ipynb`.

**All ML code is embedded below — no file uploads required.**

Key improvement: if `manifest.csv` is missing (e.g. Colab timeout during
generation), the pipeline auto-builds it by scanning sensor files and
classifying each scenario from its `fault_code` bitmask.

## Step 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu}  ({mem:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected. Training will be slow on CPU.")
    print("Go to Runtime -> Change runtime type -> T4 GPU")


## Step 2 — Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install pandas pyarrow pyyaml tqdm matplotlib seaborn -q
print("All dependencies installed.")


## Step 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")


## Step 4 — Configuration

Set `DRIVE_DATA_DIR` to where the dataset was saved by the generation notebook.

In [ ]:
import os, torch

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_DATA_DIR    = '/content/drive/MyDrive/topdrive_ai/synthetic_v2'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/topdrive_ai/results'
LOCAL_RESULTS_DIR = '/content/results'

# ── Training settings ──────────────────────────────────────────────────────
ARCHITECTURE  = 'InceptionTime'   # 'InceptionTime' or 'ResNet'
MAX_EPOCHS    = 100
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
ENSEMBLE_SIZE = 5                  # Number of InceptionTime members
SMOKE_TEST    = False              # True = 5 epochs only (fast debug run)

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f"Data dir     : {DRIVE_DATA_DIR}")
print(f"Results dir  : {LOCAL_RESULTS_DIR}")
print(f"Architecture : {ARCHITECTURE}")
print(f"Device       : {DEVICE}")
print(f"Smoke test   : {SMOKE_TEST}")


## Step 5 — Validate Dataset

Checks that the dataset directory exists and shows what’s available.
If `manifest.csv` is missing, the pipeline will auto-build it in Step 7.

In [ ]:
import os
from pathlib import Path

data_dir = Path(DRIVE_DATA_DIR)
print(f"Dataset directory: {data_dir}")
print(f"Exists: {data_dir.exists()}\n")

if not data_dir.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_dir}\n"
        "Run 01_generate_synthetic_data.ipynb first, then sync to Drive."
    )

print("Contents:")
for item in sorted(data_dir.iterdir()):
    if item.is_dir():
        n = len(list(item.iterdir()))
        print(f"  {item.name}/  ({n:,} files)")
    else:
        size = item.stat().st_size
        print(f"  {item.name}  ({size:,} bytes)")

sensor_dir = data_dir / 'sensor'
if not sensor_dir.exists() or not any(sensor_dir.iterdir()):
    raise FileNotFoundError(f"No sensor data found in {sensor_dir}")

sensor_files = sorted(sensor_dir.iterdir())
n_sensors = len(sensor_files)
fmt = sensor_files[0].suffix
print(f"\nSensor files: {n_sensors:,}  (format: {fmt})")

# Check manifest
has_manifest = (data_dir / 'manifest.parquet').exists() or \
               (data_dir / 'manifest.csv').exists()
if has_manifest:
    print("Manifest: FOUND")
else:
    print("Manifest: NOT FOUND")
    print("  -> Will auto-build from sensor fault_code data in Step 7")

# Quick peek at one file
import pandas as pd
sample = sensor_files[0]
if sample.suffix == '.parquet':
    df = pd.read_parquet(sample)
else:
    df = pd.read_csv(sample, nrows=5)
print(f"\nSample file columns ({len(df.columns)}):")
print(f"  {list(df.columns)}")
print(f"  Rows: {len(df):,}")

## Step 6 — Write ML Modules

Each cell writes one Python module to the Colab VM disk.

### features.py

In [ ]:
%%writefile features.py
"""
Feature Engineering: Derived Channels
=======================================
Computes 6 derived channels from 6 raw sensor channels.

Raw channels (from simulator sensor output):
  0: torque_ftlbs
  1: rpm
  2: pressure_psi
  3: oil_temp_f
  4: turns
  5: hookload_klbs

Derived channels (computed here):
  6: d_torque_dt         - Torque rate of change (ft-lbs/sec)
  7: d_torque_dturns     - Torque-turn slope (ft-lbs/turn)
  8: torque_norm         - Torque / target_torque (0-1+)
  9: turns_norm          - Turns / expected_turns (0-1+)
 10: phase               - Connection phase indicator (0-5)
 11: mask                - Validity mask (1=real, 0=padded)

Total: 12 channels input to the model.
"""
import numpy as np


# Channel indices in raw array
CH_TORQUE = 0
CH_RPM = 1
CH_PRESSURE = 2
CH_OIL_TEMP = 3
CH_TURNS = 4
CH_HOOKLOAD = 5

# Derived channel indices (appended after raw)
CH_D_TORQUE_DT = 6
CH_D_TORQUE_DTURNS = 7
CH_TORQUE_NORM = 8
CH_TURNS_NORM = 9
CH_PHASE = 10
CH_MASK = 11

NUM_RAW_CHANNELS = 6
NUM_DERIVED_CHANNELS = 6
NUM_TOTAL_CHANNELS = 12

# Clamp range for d_torque_dturns to prevent outliers
SLOPE_CLIP_MIN = -1e4
SLOPE_CLIP_MAX = 1e4

# Phase mapping: ConnectionState enum int -> simplified phase 0-5
_STATE_TO_PHASE = np.zeros(14, dtype=np.float32)
_STATE_TO_PHASE[0] = 0   # IDLE
_STATE_TO_PHASE[1] = 0   # APPROACH
_STATE_TO_PHASE[2] = 1   # SPIN_IN
_STATE_TO_PHASE[3] = 2   # SHOULDER
_STATE_TO_PHASE[4] = 3   # POWER_TIGHT
_STATE_TO_PHASE[5] = 3   # HOLD
_STATE_TO_PHASE[6] = 1   # BREAKOUT (reverse)
_STATE_TO_PHASE[7] = 1   # BACKOFF (reverse)
_STATE_TO_PHASE[8] = 5   # FAULT
_STATE_TO_PHASE[9] = 4   # COMPLETE
_STATE_TO_PHASE[10] = 5  # STALL
_STATE_TO_PHASE[11] = 5  # E_STOP
_STATE_TO_PHASE[12] = 5  # FAULT_RECOVERY
_STATE_TO_PHASE[13] = 1  # HANDOFF


def compute_d_torque_dt(torque: np.ndarray, dt: float = 0.01, k: int = 2) -> np.ndarray:
    """Central difference torque rate: (T[i+k] - T[i-k]) / (2k * dt).

    Vectorized numpy implementation.

    Args:
        torque: 1D array of torque values [N].
        dt: Timestep in seconds (default 0.01 = 100Hz).
        k: Half-window for central difference.

    Returns:
        1D array [N] of torque rate (ft-lbs/sec).
    """
    n = len(torque)
    result = np.zeros(n, dtype=np.float32)

    if n <= 2 * k:
        return result

    # Central difference for interior points
    denominator = 2.0 * k * dt
    result[k:n-k] = (torque[2*k:] - torque[:n-2*k]) / denominator

    # Forward difference for leading edge
    result[:k] = (torque[k:2*k] - torque[:k]) / (k * dt)

    # Backward difference for trailing edge
    result[n-k:] = (torque[n-k:] - torque[n-2*k:n-k]) / (k * dt)

    return result


def compute_d_torque_dturns(torque: np.ndarray, turns: np.ndarray,
                             k: int = 5, eps: float = 1e-6) -> np.ndarray:
    """Torque-turn slope: (T[i+k] - T[i-k]) / (turns[i+k] - turns[i-k]).

    THE diagnostic feature per Section 3.2:
      - Galling: progressive slope increase
      - Stripped thread: slope plateau/drop
      - Cross-thread: extreme slope at low turns

    Vectorized numpy implementation with zero-division protection.

    Args:
        torque: 1D array [N].
        turns: 1D array [N].
        k: Half-window (default 5 = 0.05 turns at typical RPM).
        eps: Epsilon for zero-division protection.

    Returns:
        1D array [N], clipped to [-1e4, 1e4].
    """
    n = len(torque)
    result = np.zeros(n, dtype=np.float32)

    if n <= 2 * k:
        return result

    d_torque = torque[2*k:] - torque[:n-2*k]
    d_turns = turns[2*k:] - turns[:n-2*k]

    # Safe division: only divide where d_turns is nonzero
    safe_mask = np.abs(d_turns) > eps
    result[k:n-k] = np.where(safe_mask, d_torque / np.where(safe_mask, d_turns, 1.0), 0.0)

    return np.clip(result, SLOPE_CLIP_MIN, SLOPE_CLIP_MAX)


def compute_phase_indicator(connection_state: np.ndarray) -> np.ndarray:
    """Map connection_state enum to simplified phase indicator (0-5).

    Vectorized lookup table implementation.

    Args:
        connection_state: 1D int array [N] of ConnectionState values.

    Returns:
        1D float32 array [N] with phase 0-5.
    """
    # Clip to valid range and use lookup table
    clipped = np.clip(connection_state.astype(np.int32), 0, len(_STATE_TO_PHASE) - 1)
    return _STATE_TO_PHASE[clipped]


def compute_derived_channels(raw: np.ndarray,
                              connection_state: np.ndarray,
                              target_torque: float,
                              expected_turns: float,
                              dt: float = 0.01) -> np.ndarray:
    """Compute all 6 derived channels and append to raw channels.

    Args:
        raw: Array [N, 6] of raw sensor channels.
        connection_state: 1D int array [N] from sensor data.
        target_torque: Optimum makeup torque for this pipe (ft-lbs).
        expected_turns: Expected turns-to-shoulder for this pipe.
        dt: Timestep (seconds).

    Returns:
        Array [N, 12] with raw + derived channels.
    """
    n = raw.shape[0]
    torque = raw[:, CH_TORQUE]
    turns = raw[:, CH_TURNS]

    # Derived channels (all vectorized)
    d_torque_dt = compute_d_torque_dt(torque, dt=dt)
    d_torque_dturns = compute_d_torque_dturns(torque, turns)
    torque_norm = torque / max(target_torque, 1.0)
    turns_norm = turns / max(expected_turns, 0.1)
    phase = compute_phase_indicator(connection_state)
    mask = np.ones(n, dtype=np.float32)  # All real data; padding sets to 0 later

    # Stack all 12 channels
    derived = np.stack([
        d_torque_dt,
        d_torque_dturns,
        torque_norm.astype(np.float32),
        turns_norm.astype(np.float32),
        phase,
        mask,
    ], axis=1)

    return np.concatenate([raw.astype(np.float32), derived], axis=1)


### dataset.py

In [ ]:
%%writefile dataset.py
"""
Dataset & Preprocessing Pipeline for Phase 1
===============================================
Loads Parquet/CSV scenarios, computes derived features, applies sliding windows,
normalizes, and yields (X, y) for PyTorch training.

KEY FIX: Per-window label assignment using fault_code activity.
Windows from fault scenarios are only labeled as faults if fault_code
is active in >= threshold of the window's timesteps. Pre-fault windows
(which look normal) get labeled as class 0.
"""
import json
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

import torch
from torch.utils.data import Dataset

from features import (
    compute_derived_channels,
    NUM_RAW_CHANNELS, NUM_TOTAL_CHANNELS,
    CH_MASK,
)

logger = logging.getLogger(__name__)

RAW_COLUMNS = [
    'torque_ftlbs', 'rpm', 'pressure_psi',
    'oil_temp_f', 'turns', 'hookload_klbs',
]
STATE_COLUMN = 'connection_state'

# FaultCode bitmask -> fault class (priority order, first match wins)
FAULT_PRIORITY = [
    (0x0800, 1),   # CONNECTION_JUMP -> cross_thread
    (0x0400, 4),   # WASHOUT        -> over_torque
    (0x0200, 6),   # WRONG_COMPOUND -> wrong_compound
    (0x0100, 7),   # MISALIGNED_STAB-> misaligned
    (0x0080, 3),   # STRIPPED_THREAD-> stripped
    (0x0008, 2),   # GALLING        -> galling
    (0x0004, 1),   # CROSS_THREAD   -> cross_thread
    (0x0040, 1),   # STICK_SLIP     -> cross_thread
    (0x0010, 8),   # STALL          -> stall
    (0x0001, 4),   # OVER_TORQUE    -> over_torque
]

CLASS_TO_SCENARIO_TYPE = {
    0: 'normal_casing_ltc', 1: 'cross_thread', 2: 'galling',
    3: 'stripped_thread', 4: 'over_torque', 5: 'under_torque',
    6: 'wrong_compound', 7: 'misaligned_stabbing', 8: 'stall',
}


@dataclass
class NormParams:
    mean: np.ndarray
    std: np.ndarray

    def save(self, path: str):
        with open(path, 'w') as f:
            json.dump({'mean': self.mean.tolist(), 'std': self.std.tolist()}, f, indent=2)

    @classmethod
    def load(cls, path: str) -> 'NormParams':
        with open(path) as f:
            data = json.load(f)
        return cls(mean=np.array(data['mean'], dtype=np.float32),
                   std=np.array(data['std'], dtype=np.float32))


def _classify_fault_code(fault_codes, torque_values=None, target_torque=None):
    combined = int(np.bitwise_or.reduce(fault_codes.astype(np.int64)))
    if combined == 0:
        return 0
    fault_class = 0
    for bit, cls in FAULT_PRIORITY:
        if combined & bit:
            fault_class = cls
            break
    if (fault_class == 8 and torque_values is not None
            and target_torque and target_torque > 0):
        if float(np.max(np.abs(torque_values))) < 0.65 * target_torque:
            return 5
    return fault_class


def _estimate_expected_turns(target_torque):
    if target_torque < 2000: return 8.0
    elif target_torque < 4000: return 5.5
    elif target_torque < 7000: return 4.0
    elif target_torque < 15000: return 3.5
    else: return 6.0


def auto_build_manifest(sensor_dir, class_map):
    class_to_type = {}
    for stype, cls in class_map.items():
        if cls not in class_to_type:
            class_to_type[cls] = stype
    for cls, stype in CLASS_TO_SCENARIO_TYPE.items():
        if cls not in class_to_type:
            class_to_type[cls] = stype

    files = sorted(f for f in sensor_dir.iterdir() if f.suffix in ('.parquet', '.csv'))
    if not files:
        raise FileNotFoundError(f"No sensor files in {sensor_dir}")

    logger.info(f"Auto-building manifest from {len(files)} sensor files...")
    records = []
    for i, fp in enumerate(files):
        if (i + 1) % 500 == 0 or i == 0:
            logger.info(f"  Scanning {i+1}/{len(files)}...")
        try:
            df = pd.read_parquet(fp) if fp.suffix == '.parquet' else pd.read_csv(fp)
        except Exception as e:
            logger.warning(f"  Skipping {fp.name}: {e}")
            continue

        target_torque = 5000.0
        if 'target_torque' in df.columns:
            tt = df['target_torque'].dropna()
            if len(tt) > 0:
                modes = tt.mode()
                target_torque = float(modes.iloc[0]) if len(modes) > 0 else float(tt.median())

        torque_values = df['torque_ftlbs'].values if 'torque_ftlbs' in df.columns else None
        if 'fault_code' in df.columns:
            fault_class = _classify_fault_code(df['fault_code'].values, torque_values, target_torque)
        else:
            fault_class = 0

        records.append({
            'scenario_id': i, 'filename': fp.name,
            'scenario_type': class_to_type.get(fault_class, 'normal_casing_ltc'),
            'fault_class': fault_class,
            'target_torque_ftlbs': target_torque,
            'expected_turns': _estimate_expected_turns(target_torque),
            'num_samples': len(df),
        })

    manifest = pd.DataFrame(records)
    logger.info(f"Auto-manifest: {len(manifest)} scenarios")
    for cls in sorted(manifest['fault_class'].unique()):
        count = int((manifest['fault_class'] == cls).sum())
        logger.info(f"  Class {cls} ({class_to_type.get(cls, '?')}): {count}")

    try:
        manifest.to_csv(sensor_dir.parent / 'manifest.csv', index=False)
    except OSError:
        pass
    return manifest


# ═══════════════════════════════════════════════════════════════════
# Core pipeline
# ═══════════════════════════════════════════════════════════════════

def scenario_to_windows(filepath, target_torque, expected_turns,
                         window_size=2000, stride=1000, dt=0.01):
    """Load scenario and create sliding windows.

    Returns (features, fault_codes):
      features: [W, window_size, 12] float32
      fault_codes: [W, window_size] int64 or None
    """
    try:
        df = pd.read_parquet(filepath) if filepath.suffix == '.parquet' else pd.read_csv(filepath)
    except Exception as e:
        logger.warning(f"Error loading {filepath}: {e}")
        return np.empty((0, window_size, NUM_TOTAL_CHANNELS), dtype=np.float32), None

    for col in RAW_COLUMNS + [STATE_COLUMN]:
        if col not in df.columns:
            logger.warning(f"Missing column {col} in {filepath}")
            return np.empty((0, window_size, NUM_TOTAL_CHANNELS), dtype=np.float32), None

    raw = df[RAW_COLUMNS].values.astype(np.float32)
    state = df[STATE_COLUMN].values.astype(np.int32)
    has_fc = 'fault_code' in df.columns
    fc_raw = df['fault_code'].values.astype(np.int64) if has_fc else None

    features = compute_derived_channels(raw, state, target_torque, expected_turns, dt=dt)
    n = features.shape[0]

    feat_wins, fc_wins = [], []
    start = 0
    while start < n:
        end = start + window_size
        if end <= n:
            feat_wins.append(features[start:end])
            if fc_raw is not None:
                fc_wins.append(fc_raw[start:end])
        else:
            win = np.zeros((window_size, NUM_TOTAL_CHANNELS), dtype=np.float32)
            actual = n - start
            win[:actual] = features[start:n]
            win[actual:, CH_MASK] = 0.0
            feat_wins.append(win)
            if fc_raw is not None:
                fc_w = np.zeros(window_size, dtype=np.int64)
                fc_w[:actual] = fc_raw[start:n]
                fc_wins.append(fc_w)
        start += stride

    if not feat_wins:
        win = np.zeros((window_size, NUM_TOTAL_CHANNELS), dtype=np.float32)
        actual = min(n, window_size)
        win[:actual] = features[:actual]
        win[actual:, CH_MASK] = 0.0
        feat_wins.append(win)
        if fc_raw is not None:
            fc_w = np.zeros(window_size, dtype=np.int64)
            fc_w[:actual] = fc_raw[:actual]
            fc_wins.append(fc_w)

    return np.stack(feat_wins), (np.stack(fc_wins) if fc_wins else None)


def build_dataset_index(manifest, scenario_ids, sensor_dir, class_map,
                         window_size=2000, stride=1000,
                         fault_threshold=0.10):
    """Build windowed dataset with PER-WINDOW label assignment.

    For fault scenarios, each window is labeled based on whether fault_code
    is active in >= fault_threshold fraction of timesteps. Pre-fault windows
    are relabeled as normal (class 0).
    """
    all_windows, all_labels, all_sids = [], [], []
    relabeled, total_fault = 0, 0

    for _, row in manifest[manifest['scenario_id'].isin(scenario_ids)].iterrows():
        fp = sensor_dir / row['filename']
        if not fp.exists():
            continue

        tt = float(row.get('target_torque_ftlbs', 5000))
        et = float(row.get('expected_turns', 5.0))

        if 'fault_class' in row.index and pd.notna(row['fault_class']):
            sc = int(row['fault_class'])
        else:
            sc = class_map.get(row['scenario_type'], 0)

        windows, fc_windows = scenario_to_windows(fp, tt, et, window_size, stride)
        if windows.shape[0] == 0:
            continue

        nw = windows.shape[0]

        if sc == 0 or fc_windows is None:
            labels = [sc] * nw
        else:
            labels = []
            for wi in range(nw):
                total_fault += 1
                frac = np.count_nonzero(fc_windows[wi]) / len(fc_windows[wi])
                if frac >= fault_threshold:
                    labels.append(sc)
                else:
                    labels.append(0)
                    relabeled += 1

        all_windows.append(windows)
        all_labels.extend(labels)
        all_sids.extend([int(row['scenario_id'])] * nw)

    if total_fault > 0:
        logger.info(f"  Per-window labeling: {relabeled}/{total_fault} fault windows "
                     f"relabeled as normal ({100*relabeled/total_fault:.1f}% were pre-fault)")

    return all_windows, np.array(all_labels), np.array(all_sids)


def compute_norm_params(windows_list):
    all_data = np.concatenate([w.reshape(-1, NUM_TOTAL_CHANNELS) for w in windows_list], axis=0)
    mean = np.mean(all_data, axis=0).astype(np.float32)
    std = np.std(all_data, axis=0).astype(np.float32)
    std = np.where(std < 1e-8, 1.0, std)
    return NormParams(mean=mean, std=std)


def apply_normalization(windows, norm):
    normalized = (windows - norm.mean) / norm.std
    normalized[:, :, CH_MASK] = windows[:, :, CH_MASK]
    return normalized.astype(np.float32)


class Phase1Dataset(Dataset):
    def __init__(self, windows, labels):
        self.windows = windows
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        x = torch.from_numpy(self.windows[idx]).T
        return x, int(self.labels[idx])


def create_splits(manifest, class_map, split_ratio=(0.70, 0.15, 0.15), seed=42):
    rng = np.random.RandomState(seed)
    manifest = manifest.copy()
    manifest['fault_class'] = manifest['scenario_type'].map(class_map).fillna(0).astype(int)
    class_groups = manifest.groupby('fault_class')['scenario_id'].apply(list).to_dict()

    train_ids, val_ids, test_ids = [], [], []
    train_frac, val_frac, _ = split_ratio
    for cls, ids in class_groups.items():
        ids = np.array(ids)
        rng.shuffle(ids)
        n = len(ids)
        n_train = max(1, int(n * train_frac))
        n_val = max(1, int(n * val_frac))
        train_ids.extend(ids[:n_train].tolist())
        val_ids.extend(ids[n_train:n_train + n_val].tolist())
        test_ids.extend(ids[n_train + n_val:].tolist())
    return np.array(train_ids), np.array(val_ids), np.array(test_ids)


def prepare_datasets(dataset_dir, class_map, window_size=2000, stride=1000,
                      split_ratio=(0.70, 0.15, 0.15), split_seed=42,
                      norm_params_path=None, fault_threshold=0.10):
    """Full pipeline: manifest -> split -> window -> normalize -> Dataset."""
    dataset_dir = Path(dataset_dir)
    sensor_dir = dataset_dir / 'sensor'

    manifest_parquet = dataset_dir / 'manifest.parquet'
    manifest_csv = dataset_dir / 'manifest.csv'

    if manifest_parquet.exists():
        manifest = pd.read_parquet(manifest_parquet)
    elif manifest_csv.exists():
        manifest = pd.read_csv(manifest_csv)
    else:
        manifest = auto_build_manifest(sensor_dir, class_map)

    if 'scenario_id' not in manifest.columns:
        manifest['scenario_id'] = range(len(manifest))

    train_ids, val_ids, test_ids = create_splits(manifest, class_map, split_ratio, split_seed)

    print(f"Building training windows ({len(train_ids)} scenarios)...")
    train_windows, train_labels, _ = build_dataset_index(
        manifest, train_ids, sensor_dir, class_map, window_size, stride, fault_threshold)

    print(f"Building validation windows ({len(val_ids)} scenarios)...")
    val_windows, val_labels, _ = build_dataset_index(
        manifest, val_ids, sensor_dir, class_map, window_size, stride, fault_threshold)

    print(f"Building test windows ({len(test_ids)} scenarios)...")
    test_windows, test_labels, _ = build_dataset_index(
        manifest, test_ids, sensor_dir, class_map, window_size, stride, fault_threshold)

    train_all = np.concatenate(train_windows) if train_windows else np.empty((0, window_size, NUM_TOTAL_CHANNELS))
    val_all = np.concatenate(val_windows) if val_windows else np.empty((0, window_size, NUM_TOTAL_CHANNELS))
    test_all = np.concatenate(test_windows) if test_windows else np.empty((0, window_size, NUM_TOTAL_CHANNELS))

    print("Computing normalization from training data...")
    norm = compute_norm_params(train_windows if train_windows else [train_all])
    train_all = apply_normalization(train_all, norm)
    val_all = apply_normalization(val_all, norm)
    test_all = apply_normalization(test_all, norm)

    if norm_params_path:
        norm.save(norm_params_path)

    # Check for NaN/Inf
    for name, arr in [('train', train_all), ('val', val_all), ('test', test_all)]:
        if np.any(np.isnan(arr)):
            print(f"  WARNING: {name} has NaN values!")
        if np.any(np.isinf(arr)):
            print(f"  WARNING: {name} has Inf values!")

    print(f"\nDataset sizes: train={len(train_labels)}, val={len(val_labels)}, test={len(test_labels)}")
    from collections import Counter
    for sn, lb in [('train', train_labels), ('val', val_labels), ('test', test_labels)]:
        dist = Counter(int(l) for l in lb)
        print(f"  {sn}: {dict(sorted(dist.items()))}")

    return (Phase1Dataset(train_all, train_labels),
            Phase1Dataset(val_all, val_labels),
            Phase1Dataset(test_all, test_labels), norm)

### models.py

In [ ]:
%%writefile models.py
"""
Model Architectures for Phase 1 Pretraining
=============================================

Two architectures:
  1. ResNet baseline (Tier 1) — validates data pipeline, ~50K params, 30 min on GPU
  2. InceptionTime (Tier 2) — primary model, ensemble of 5, ~250K total params

Per Section 4 of the Phase 1 spec:
  - Input: [B, 12, 2000] (12 channels, 2000 timesteps)
  - Output: [B, 10] (10 classes)
  - InceptionTime: 6 Inception modules in 2 residual blocks of 3
  - Ensemble: 5 independently-initialized networks, mean of softmax outputs
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional


# ═══════════════════════════════════════════════════════════════════
# ResNet Baseline (Tier 1)
# ═══════════════════════════════════════════════════════════════════

class ResBlock1d(nn.Module):
    """1D Residual block: Conv-BN-ReLU-Conv-BN + shortcut."""

    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size_1: int = 7, kernel_size_2: int = 5):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size_1,
                               padding=kernel_size_1 // 2)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size_2,
                               padding=kernel_size_2 // 2)
        self.bn2 = nn.BatchNorm1d(out_channels)

        # Shortcut for dimension mismatch
        self.shortcut = nn.Identity()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)


class ResNetBaseline(nn.Module):
    """Simple 3-block ResNet for time series classification.

    ~50K parameters. Target: >80% macro F1.
    If this fails, debug the data pipeline before trying InceptionTime.

    Args:
        c_in: Input channels (default 12).
        c_out: Output classes (default 10).
    """

    def __init__(self, c_in: int = 12, c_out: int = 10):
        super().__init__()
        self.block1 = ResBlock1d(c_in, 64)
        self.block2 = ResBlock1d(64, 128)
        self.block3 = ResBlock1d(128, 128)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, c_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, c_in, T] input tensor (channels-first).
        Returns:
            [B, c_out] logits (before softmax).
        """
        out = self.block1(x)
        out = self.block2(out)
        out = self.block3(out)
        out = self.gap(out).squeeze(-1)  # [B, 128]
        return self.fc(out)


# ═══════════════════════════════════════════════════════════════════
# InceptionTime (Tier 2, Primary)
# ═══════════════════════════════════════════════════════════════════

class InceptionModule(nn.Module):
    """Single Inception module with multi-scale parallel convolutions.

    Architecture per Section 4.2.1:
      Bottleneck -> 3 parallel conv branches (k=10, k=20, k=40) + MaxPool branch
      -> Concat -> BatchNorm -> ReLU

    Args:
        in_channels: Input channel count.
        nf: Number of filters per branch (default 32).
        bottleneck_size: Bottleneck output channels (default 32).
    """

    def __init__(self, in_channels: int, nf: int = 32,
                 bottleneck_size: int = 32):
        super().__init__()

        # Bottleneck: reduce channels before parallel convs
        self.bottleneck = nn.Conv1d(in_channels, bottleneck_size, kernel_size=1, bias=False)

        # Branch A: short-range patterns (k=11, ~0.11s at 100Hz)
        self.conv_a = nn.Conv1d(bottleneck_size, nf, kernel_size=11,
                                padding=5, bias=False)

        # Branch B: medium-range patterns (k=21, ~0.21s)
        self.conv_b = nn.Conv1d(bottleneck_size, nf, kernel_size=21,
                                padding=10, bias=False)

        # Branch C: long-range patterns (k=41, ~0.41s)
        self.conv_c = nn.Conv1d(bottleneck_size, nf, kernel_size=41,
                                padding=20, bias=False)

        # MaxPool branch: translation-invariant features
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=1, padding=1)
        self.conv_mp = nn.Conv1d(in_channels, nf, kernel_size=1, bias=False)

        # Output: concat of 4 branches = 4 * nf channels
        self.bn = nn.BatchNorm1d(4 * nf)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Bottleneck
        bottleneck_out = self.bottleneck(x)

        # Parallel branches
        branch_a = self.conv_a(bottleneck_out)
        branch_b = self.conv_b(bottleneck_out)
        branch_c = self.conv_c(bottleneck_out)
        branch_mp = self.conv_mp(self.maxpool(x))

        # Concat + BN + ReLU
        out = torch.cat([branch_a, branch_b, branch_c, branch_mp], dim=1)
        return F.relu(self.bn(out))


class InceptionBlock(nn.Module):
    """Residual block of 3 Inception modules with shortcut connection.

    Section 4.2.2: Each residual block = 3 Inception modules + skip connection.
    """

    def __init__(self, in_channels: int, nf: int = 32):
        super().__init__()
        out_channels = 4 * nf  # Each Inception module outputs 4*nf channels

        self.module1 = InceptionModule(in_channels, nf)
        self.module2 = InceptionModule(out_channels, nf)
        self.module3 = InceptionModule(out_channels, nf)

        # Shortcut: match dimensions if needed
        self.shortcut = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_channels),
        ) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        out = self.module1(x)
        out = self.module2(out)
        out = self.module3(out)
        return F.relu(out + residual)


class InceptionTimeNetwork(nn.Module):
    """Single InceptionTime network (one member of the ensemble).

    Architecture (Section 4.2.2):
      Input [B, 12, 2000]
      -> InceptionBlock 1 (3 modules + residual) -> [B, 128, 2000]
      -> InceptionBlock 2 (3 modules + residual) -> [B, 128, 2000]
      -> Global Average Pool -> [B, 128]
      -> FC -> [B, 10]

    ~50K parameters per network.

    Args:
        c_in: Input channels (default 12).
        c_out: Number of classes (default 10).
        nf: Filters per Inception branch (default 32). Output = 4*nf = 128.
    """

    def __init__(self, c_in: int = 12, c_out: int = 10, nf: int = 32):
        super().__init__()
        self.block1 = InceptionBlock(c_in, nf)
        self.block2 = InceptionBlock(4 * nf, nf)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4 * nf, c_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.block1(x)
        out = self.block2(out)
        out = self.gap(out).squeeze(-1)
        return self.fc(out)


class InceptionTimeEnsemble(nn.Module):
    """Ensemble of N InceptionTime networks.

    Final prediction = mean of N softmax outputs.
    Per Section 4.2.2: ensemble of 5 reduces variance by ~60%.

    Args:
        c_in: Input channels.
        c_out: Number of classes.
        nf: Filters per branch.
        ensemble_size: Number of networks (default 5).
        seeds: Random seeds for initialization (one per member).
    """

    def __init__(self, c_in: int = 12, c_out: int = 10, nf: int = 32,
                 ensemble_size: int = 5,
                 seeds: Optional[List[int]] = None):
        super().__init__()
        self.ensemble_size = ensemble_size
        seeds = seeds or list(range(ensemble_size))

        self.networks = nn.ModuleList()
        for seed in seeds:
            torch.manual_seed(seed)
            net = InceptionTimeNetwork(c_in, c_out, nf)
            self.networks.append(net)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Mean of softmax outputs from all ensemble members."""
        outputs = []
        for net in self.networks:
            logits = net(x)
            probs = F.softmax(logits, dim=-1)
            outputs.append(probs)
        return torch.stack(outputs).mean(dim=0)

    def forward_logits(self, x: torch.Tensor, member_idx: int) -> torch.Tensor:
        """Forward pass through a single ensemble member (for training)."""
        return self.networks[member_idx](x)


def create_model(architecture: str = 'InceptionTime',
                 c_in: int = 12, c_out: int = 10,
                 nf: int = 32, **kwargs) -> nn.Module:
    """Factory function to create model by name.

    Args:
        architecture: 'InceptionTime', 'InceptionTimeEnsemble', or 'ResNet'.
        c_in: Input channels.
        c_out: Output classes.
        nf: Filters per branch.

    Returns:
        nn.Module instance.
    """
    if architecture == 'ResNet':
        return ResNetBaseline(c_in, c_out)
    elif architecture == 'InceptionTime':
        return InceptionTimeNetwork(c_in, c_out, nf)
    elif architecture == 'InceptionTimeEnsemble':
        return InceptionTimeEnsemble(c_in, c_out, nf, **kwargs)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")


def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


### losses.py

In [ ]:
%%writefile losses.py
"""
Loss Functions for Phase 1 Pretraining
========================================
Focal loss with per-class alpha weighting and label smoothing.

Per Section 5.1 of the Phase 1 spec:
  - Focal loss down-weights well-classified examples via (1 - p_t)^gamma
  - Per-class alpha inversely proportional to class frequency, adjusted for safety
  - Label smoothing (0.1) prevents overconfidence on synthetic data
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional


class FocalLoss(nn.Module):
    """Focal loss with per-class alpha weighting and label smoothing.

    Args:
        alpha: Per-class weight tensor [num_classes]. Higher = more weight.
        gamma: Focusing parameter. gamma=0 -> standard CE. gamma=2 -> standard focal.
        label_smoothing: Smoothing factor (0-1). 0.1 recommended for synthetic data.
    """

    def __init__(self, alpha: List[float], gamma: float = 2.0,
                 label_smoothing: float = 0.1):
        super().__init__()
        self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: [B, num_classes] raw model output (before softmax).
            targets: [B] integer class labels.

        Returns:
            Scalar loss.
        """
        num_classes = logits.size(-1)

        # Label smoothing: spread (1-eps) on true class, eps/(K-1) on others
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits)
            smooth_targets.fill_(self.label_smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)

        # Log-probabilities
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)

        # Focal modulation: (1 - p_t)^gamma
        focal_weight = (1.0 - probs) ** self.gamma

        # Per-class alpha weighting
        alpha_weight = self.alpha[targets].unsqueeze(-1)  # [B, 1]

        # Combine: -alpha * (1-p)^gamma * smooth_target * log(p)
        loss = -alpha_weight * focal_weight * smooth_targets * log_probs
        return loss.sum(dim=-1).mean()


class MixupFocalLoss(nn.Module):
    """Focal loss adapted for mixup augmentation.

    When mixup is applied, we get two targets (y_a, y_b) and a lambda.
    Loss = lam * FocalLoss(logits, y_a) + (1 - lam) * FocalLoss(logits, y_b).
    """

    def __init__(self, alpha: List[float], gamma: float = 2.0,
                 label_smoothing: float = 0.1):
        super().__init__()
        self.focal = FocalLoss(alpha, gamma, label_smoothing)

    def forward(self, logits: torch.Tensor,
                targets_a: torch.Tensor, targets_b: torch.Tensor,
                lam: float) -> torch.Tensor:
        return lam * self.focal(logits, targets_a) + (1 - lam) * self.focal(logits, targets_b)


def mixup_data(x: torch.Tensor, y: torch.Tensor,
               alpha: float = 0.2) -> tuple:
    """Apply mixup augmentation to a batch.

    Args:
        x: Input tensor [B, C, T].
        y: Target labels [B].
        alpha: Beta distribution parameter. 0.2 = mostly pure samples.

    Returns:
        (mixed_x, y_a, y_b, lam)
    """
    if alpha > 0:
        lam = float(torch.distributions.Beta(alpha, alpha).sample())
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


### sampler.py

In [ ]:
%%writefile sampler.py
"""
Balanced Batch Sampler
========================
Ensures every mini-batch contains examples from all 10 classes.
Oversamples minority classes, undersamples majority class (normal).

Per Section 5.3: samples_per_class = batch_size // num_classes = 64 // 10 = 6.
Each batch gets ~6 samples per class = 60 samples (padded to 64 randomly).
"""
import numpy as np
import torch
from torch.utils.data import Sampler
from typing import List, Iterator, Dict, Optional
from collections import defaultdict


class BalancedBatchSampler(Sampler):
    """Yields batches with balanced class representation.

    For each batch:
      1. Sample `samples_per_class` indices from each class
      2. Shuffle the combined indices
      3. Yield them as a single batch

    Minority classes are oversampled (with replacement).
    Majority classes are undersampled (without replacement).

    Args:
        labels: List/array of integer class labels for each sample.
        batch_size: Target batch size (may be slightly different due to rounding).
        samples_per_class: How many samples per class per batch.
            Default: batch_size // num_classes.
        drop_last: Whether to drop the last incomplete batch.
    """

    def __init__(self, labels: np.ndarray, batch_size: int = 64,
                 samples_per_class: Optional[int] = None,
                 drop_last: bool = False):
        self.labels = np.asarray(labels)
        self.batch_size = batch_size
        self.drop_last = drop_last

        # Discover classes and build per-class index lists
        self.classes = sorted(set(self.labels.tolist()))
        self.num_classes = len(self.classes)
        self.samples_per_class = samples_per_class or (batch_size // self.num_classes)

        self.class_indices: Dict[int, np.ndarray] = {}
        for cls in self.classes:
            self.class_indices[cls] = np.where(self.labels == cls)[0]

        # Actual batch size = samples_per_class * num_classes
        self.actual_batch_size = self.samples_per_class * self.num_classes

        # Number of batches per epoch: cap at total_samples / actual_batch_size
        # so we see roughly each sample once (via balanced sampling).
        # Without the cap, the majority class alone could drive 500+ batches.
        total_samples = len(self.labels)
        self._num_batches = max(1, total_samples // self.actual_batch_size)

    def __iter__(self) -> Iterator[List[int]]:
        for _ in range(self._num_batches):
            batch = []
            for cls in self.classes:
                idx = self.class_indices[cls]
                # Oversample if class has fewer samples than needed
                replace = len(idx) < self.samples_per_class
                chosen = np.random.choice(idx, self.samples_per_class, replace=replace)
                batch.extend(chosen.tolist())

            np.random.shuffle(batch)
            yield batch

    def __len__(self) -> int:
        return self._num_batches


### eval.py

In [ ]:
%%writefile eval.py
"""
Evaluation Script for Phase 1
================================
Evaluates trained models on held-out test set.
"""
import argparse
import json
import csv
import logging
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from dataset import prepare_datasets, Phase1Dataset, NormParams
from models import InceptionTimeNetwork, ResNetBaseline, count_parameters

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)


def load_ensemble(checkpoint_dir, device, c_in=12, c_out=9, nf=32, ensemble_size=5):
    models = []
    for i in range(ensemble_size):
        path = checkpoint_dir / f'inception_{i}_best.pt'
        if not path.exists():
            logger.warning(f"Missing checkpoint: {path}")
            continue
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        model = InceptionTimeNetwork(c_in, c_out, nf)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device).eval()
        models.append(model)
        logger.info(f"Loaded inception_{i}: val_f1={checkpoint.get('val_macro_f1', 'N/A')}")
    return models


def ensemble_predict(models, dataloader, device):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            batch_probs = []
            for model in models:
                logits = model(X_batch)
                batch_probs.append(F.softmax(logits, dim=-1))
            avg_probs = torch.stack(batch_probs).mean(dim=0)
            all_probs.append(avg_probs.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.array(all_labels)
    all_preds = all_probs.argmax(axis=1)
    return all_preds, all_labels, all_probs


def compute_confusion_matrix(preds, labels, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for true, pred in zip(labels, preds):
        cm[true, pred] += 1
    return cm


def compute_per_class_metrics(preds, labels, num_classes):
    metrics = []
    for cls in range(num_classes):
        tp = ((preds == cls) & (labels == cls)).sum()
        fp = ((preds == cls) & (labels != cls)).sum()
        fn = ((preds != cls) & (labels == cls)).sum()
        support = (labels == cls).sum()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)
        metrics.append({
            'class': cls, 'precision': float(precision),
            'recall': float(recall), 'f1': float(f1), 'support': int(support),
        })
    return metrics


def assess_exit_criteria(per_class_metrics, class_names):
    f1_scores = [m['f1'] for m in per_class_metrics]
    macro_f1 = np.mean(f1_scores)
    normal_recall = per_class_metrics[0]['recall']
    fault_recalls = [m['recall'] for m in per_class_metrics[1:]]
    avg_fault_recall = np.mean(fault_recalls) if fault_recalls else 0.0

    criteria = [
        {'criterion': 'Macro F1 > 0.85', 'value': f'{macro_f1:.4f}',
         'target': '0.85', 'status': 'PASS' if macro_f1 > 0.85 else 'FAIL'},
        {'criterion': 'All per-class F1 > 0.70', 'value': f'min={min(f1_scores):.4f}',
         'target': '0.70', 'status': 'PASS' if min(f1_scores) > 0.70 else 'FAIL'},
        {'criterion': 'Normal class recall > 0.95', 'value': f'{normal_recall:.4f}',
         'target': '0.95', 'status': 'PASS' if normal_recall > 0.95 else 'FAIL'},
        {'criterion': 'Avg fault recall > 0.80', 'value': f'{avg_fault_recall:.4f}',
         'target': '0.80', 'status': 'PASS' if avg_fault_recall > 0.80 else 'FAIL'},
    ]
    for i, (m, name) in enumerate(zip(per_class_metrics, class_names)):
        criteria.append({
            'criterion': f'  {name} F1 > 0.70', 'value': f'{m["f1"]:.4f}',
            'target': '0.70', 'status': 'PASS' if m['f1'] > 0.70 else 'FAIL',
        })
    return criteria


def save_confusion_matrix_csv(cm, class_names, path):
    with open(path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([''] + class_names)
        for i, name in enumerate(class_names):
            writer.writerow([name] + cm[i].tolist())


def save_confusion_matrix_text(cm, class_names):
    lines = []
    header = f"{'':>20s}" + ''.join(f'{n:>12s}' for n in class_names)
    lines.append(header)
    lines.append('-' * len(header))
    for i, name in enumerate(class_names):
        row = f'{name:>20s}'
        for j in range(len(class_names)):
            pct = cm[i, j] / max(cm[i].sum(), 1) * 100
            row += f'{cm[i, j]:>8d}({pct:4.1f}%)'
        lines.append(row)
    return '\n'.join(lines)


def generate_report(results_dir, class_names, macro_f1, per_class, criteria, cm,
                     ensemble_size, model_label='InceptionTime ensemble'):
    lines = [
        '# Phase 1 Evaluation Report', '',
        f'**Model**: {model_label}', f'**Ensemble size**: {ensemble_size}',
        f'**Macro F1**: {macro_f1:.4f}', '',
        '## Exit Criteria', '| Criterion | Value | Target | Status |',
        '|-----------|-------|--------|--------|',
    ]
    for c in criteria:
        lines.append(f"| {c['criterion']} | {c['value']} | {c['target']} | {c['status']} |")

    lines.extend(['', '## Per-Class Metrics',
        '| Class | Precision | Recall | F1 | Support |',
        '|-------|-----------|--------|-----|---------|'])
    for m, name in zip(per_class, class_names):
        lines.append(f"| {name} | {m['precision']:.4f} | {m['recall']:.4f} | {m['f1']:.4f} | {m['support']} |")

    lines.extend(['', '## Confusion Matrix', '```',
                  save_confusion_matrix_text(cm, class_names), '```'])

    report_path = results_dir / 'phase1_report.md'
    with open(report_path, 'w') as f:
        f.write('\n'.join(lines))
    print(f"Report saved to {report_path}")


def main():
    parser = argparse.ArgumentParser(description='Phase 1 Evaluation')
    parser.add_argument('--config', type=str, default='config.yaml')
    parser.add_argument('--checkpoint-dir', type=str, default='results/checkpoints')
    parser.add_argument('--model', type=str, default=None)
    parser.add_argument('--output-dir', type=str, default='./results')
    args = parser.parse_args()

    with open(args.config) as f:
        config = yaml.safe_load(f)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    class_names = config['class_names']
    num_classes = len(class_names)

    logger.info("Loading test dataset...")
    dataset_dir = config['data']['dataset_dir']
    if not Path(dataset_dir).is_absolute():
        dataset_dir = str(Path(__file__).parent / dataset_dir)

    _, _, test_dataset, _ = prepare_datasets(
        dataset_dir=dataset_dir, class_map=config['class_map'],
        window_size=config['data']['window_size'],
        stride=config['data']['window_stride'],
        split_ratio=tuple(config['data']['split_ratio']),
        split_seed=config['data'].get('split_seed', 42))

    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

    arch = config['model']['architecture']
    c_in, c_out, nf = config['model']['c_in'], config['model']['c_out'], config['model']['nf']

    if arch == 'ResNet':
        ckpt_path = Path(args.checkpoint_dir) / 'resnet_baseline.pt'
        checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = ResNetBaseline(c_in, c_out)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device).eval()
        models = [model]
    else:
        ensemble_size = config['model'].get('ensemble_size', 5)
        models = load_ensemble(Path(args.checkpoint_dir), device, c_in, c_out, nf, ensemble_size)

    preds, labels, probs = ensemble_predict(models, test_loader, device)

    cm = compute_confusion_matrix(preds, labels, num_classes)
    per_class = compute_per_class_metrics(preds, labels, num_classes)
    macro_f1 = np.mean([m['f1'] for m in per_class])
    criteria = assess_exit_criteria(per_class, class_names)

    print(f"\n{'='*60}")
    print(f"Macro F1: {macro_f1:.4f}")
    for c in criteria:
        marker = '[PASS]' if c['status'] == 'PASS' else '[FAIL]'
        print(f"  {marker} {c['criterion']}: {c['value']} (target: {c['target']})")

    generate_report(output_dir, class_names, macro_f1, per_class, criteria, cm, len(models))


if __name__ == '__main__':
    main()

### train.py

In [ ]:
%%writefile train.py
"""
Training Loop for Phase 1
============================
Key improvements over original:
  - num_classes computed from config (was hardcoded to 10)
  - LR warmup (linear over first N epochs) before cosine decay
  - CrossEntropyLoss by default (simpler, more stable gradients)
  - Mixup disabled by default (reduce over-regularization)
  - Per-class F1 logged each epoch for monitoring
"""
import argparse
import json
import time
import csv
import logging
import math
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import yaml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW

from tqdm import tqdm

from dataset import prepare_datasets, Phase1Dataset
from models import create_model, InceptionTimeNetwork, count_parameters
from losses import FocalLoss, MixupFocalLoss, mixup_data
from sampler import BalancedBatchSampler

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)


def get_lr(epoch, warmup_epochs, max_epochs, base_lr, eta_min=1e-6):
    """Linear warmup then cosine decay."""
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(max_epochs - warmup_epochs, 1)
    return eta_min + 0.5 * (base_lr - eta_min) * (1 + math.cos(math.pi * progress))


def evaluate(model, dataloader, device, num_classes):
    """Evaluate model. num_classes is required (no default)."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    per_class_f1 = []
    for cls in range(num_classes):
        tp = ((all_preds == cls) & (all_labels == cls)).sum()
        fp = ((all_preds == cls) & (all_labels != cls)).sum()
        fn = ((all_preds != cls) & (all_labels == cls)).sum()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)
        per_class_f1.append(f1)

    return {
        'accuracy': float((all_preds == all_labels).mean()),
        'macro_f1': float(np.mean(per_class_f1)),
        'per_class_f1': [float(f) for f in per_class_f1],
    }


def train_single_model(model, train_dataset, val_dataset, config, device,
                        checkpoint_path, log_path, model_name="model"):
    """Train a single model (one ensemble member or baseline).

    Returns best validation macro F1.
    """
    tc = config['training']
    lc = config['loss']
    num_classes = config['model']['c_out']

    pin_mem = device.type == 'cuda'
    num_workers = config['data'].get('num_workers', 0)

    # Balanced batch sampler
    train_sampler = BalancedBatchSampler(
        train_dataset.labels,
        batch_size=tc['batch_size'],
        samples_per_class=config['sampler']['samples_per_class'],
    )
    train_loader = DataLoader(
        train_dataset, batch_sampler=train_sampler,
        num_workers=num_workers, pin_memory=pin_mem,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=tc['batch_size'] * 2,
        shuffle=False, num_workers=num_workers, pin_memory=pin_mem,
    )

    # Optimizer
    optimizer = AdamW(model.parameters(), lr=tc['lr'], weight_decay=tc['weight_decay'])

    # Loss: CE with class weights (default) or Focal
    loss_type = lc.get('type', 'CrossEntropyLoss')
    if loss_type == 'FocalLoss':
        criterion = FocalLoss(
            alpha=lc['alpha'], gamma=lc['gamma'],
            label_smoothing=lc.get('label_smoothing', 0.0),
        ).to(device)
        mixup_criterion = MixupFocalLoss(
            alpha=lc['alpha'], gamma=lc['gamma'],
            label_smoothing=lc.get('label_smoothing', 0.0),
        ).to(device)
    else:
        # Standard CrossEntropyLoss with optional class weights
        alpha = lc.get('alpha', None)
        weight = torch.tensor(alpha, dtype=torch.float32).to(device) if alpha else None
        criterion = nn.CrossEntropyLoss(
            weight=weight,
            label_smoothing=lc.get('label_smoothing', 0.0),
        )
        mixup_criterion = None  # Mixup with CE not supported here

    # Mixed precision
    use_amp = tc.get('mixed_precision', False) and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    # Training state
    best_f1 = 0.0
    patience_counter = 0
    patience = tc.get('early_stop_patience', 15)
    max_epochs = tc['max_epochs']
    warmup_epochs = tc.get('warmup_epochs', 5)
    grad_clip = tc.get('gradient_clip', 1.0)
    mixup_alpha = config['augmentation'].get('mixup_alpha', 0.2)
    mixup_prob = config['augmentation'].get('mixup_prob', 0.0)

    log_rows = []

    logger.info(f"Training {model_name}: {count_parameters(model):,} params, "
                f"{len(train_dataset)} train, {len(val_dataset)} val, "
                f"loss={loss_type}, warmup={warmup_epochs} epochs")

    for epoch in range(max_epochs):
        epoch_start = time.monotonic()

        # LR schedule: warmup + cosine
        lr = get_lr(epoch, warmup_epochs, max_epochs, tc['lr'],
                     tc.get('scheduler_eta_min', 1e-6))
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        # ── Train ──
        model.train()
        epoch_loss = 0.0
        num_batches = 0

        pbar = tqdm(train_loader,
                     desc=f"[{model_name}] Epoch {epoch+1:3d}/{max_epochs}",
                     leave=False, ncols=100)
        for X_batch, y_batch in pbar:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            # Optional mixup (only with Focal loss)
            use_mixup = (mixup_prob > 0 and mixup_criterion is not None
                         and np.random.random() < mixup_prob)
            if use_mixup:
                X_batch, y_a, y_b, lam = mixup_data(X_batch, y_batch, mixup_alpha)

            if use_amp:
                with torch.amp.autocast('cuda'):
                    logits = model(X_batch)
                    if use_mixup:
                        loss = mixup_criterion(logits, y_a, y_b, lam)
                    else:
                        loss = criterion(logits, y_batch)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(X_batch)
                if use_mixup:
                    loss = mixup_criterion(logits, y_a, y_b, lam)
                else:
                    loss = criterion(logits, y_batch)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1
            pbar.set_postfix(loss=f"{epoch_loss / num_batches:.4f}", lr=f"{lr:.2e}")

        pbar.close()
        avg_loss = epoch_loss / max(num_batches, 1)

        # ── Validate ──
        val_metrics = evaluate(model, val_loader, device, num_classes=num_classes)
        val_f1 = val_metrics['macro_f1']

        epoch_time = time.monotonic() - epoch_start

        log_row = {
            'epoch': epoch + 1, 'train_loss': avg_loss,
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_f1, 'lr': lr, 'time_s': epoch_time,
        }
        # Also log per-class F1
        for ci, f1 in enumerate(val_metrics['per_class_f1']):
            log_row[f'val_f1_class_{ci}'] = f1
        log_rows.append(log_row)

        # Early stopping
        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_macro_f1': best_f1,
                'config': config,
            }, checkpoint_path)
        else:
            patience_counter += 1

        # Log per-class F1 for first few epochs and then every 10
        pcf1_str = ""
        if epoch < 3 or (epoch + 1) % 10 == 0 or patience_counter >= patience:
            pcf1_str = " | per_class=[" + ",".join(f"{f:.2f}" for f in val_metrics['per_class_f1']) + "]"

        logger.info(
            f"  [{model_name}] E{epoch+1:3d}/{max_epochs} | "
            f"loss={avg_loss:.4f} | val_f1={val_f1:.4f} | "
            f"best={best_f1:.4f} | pat={patience_counter}/{patience} | "
            f"lr={lr:.2e} | {epoch_time:.1f}s{pcf1_str}"
        )

        if patience_counter >= patience:
            logger.info(f"  [{model_name}] Early stopping at epoch {epoch+1}")
            break

    if log_rows:
        with open(log_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=log_rows[0].keys())
            writer.writeheader()
            writer.writerows(log_rows)

    logger.info(f"  [{model_name}] Best val F1: {best_f1:.4f}")
    return best_f1


def main():
    parser = argparse.ArgumentParser(description='Phase 1 Training')
    parser.add_argument('--config', type=str, default='config.yaml')
    parser.add_argument('--model', type=str, default=None)
    parser.add_argument('--epochs', type=int, default=None)
    parser.add_argument('--smoke-test', action='store_true')
    parser.add_argument('--output-dir', type=str, default='./results')
    parser.add_argument('--device', type=str, default=None)
    args = parser.parse_args()

    config_path = Path(args.config)
    if not config_path.is_absolute():
        config_path = Path(__file__).parent / config_path
    with open(config_path) as f:
        config = yaml.safe_load(f)

    if args.model:
        config['model']['architecture'] = args.model
    if args.epochs:
        config['training']['max_epochs'] = args.epochs
    if args.smoke_test:
        config['training']['max_epochs'] = 5
        config['training']['early_stop_patience'] = 3

    if args.device:
        device = torch.device(args.device)
    elif torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')
    logger.info(f"Device: {device}")

    output_dir = Path(args.output_dir)
    checkpoints_dir = output_dir / 'checkpoints'
    logs_dir = output_dir / 'logs'
    checkpoints_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    seed = config['experiment']['seed']
    torch.manual_seed(seed)
    np.random.seed(seed)

    dataset_dir = config['data']['dataset_dir']
    if not Path(dataset_dir).is_absolute():
        dataset_dir = str(Path(__file__).parent / dataset_dir)

    train_dataset, val_dataset, test_dataset, norm_params = prepare_datasets(
        dataset_dir=dataset_dir, class_map=config['class_map'],
        window_size=config['data']['window_size'],
        stride=config['data']['window_stride'],
        split_ratio=tuple(config['data']['split_ratio']),
        split_seed=config['data'].get('split_seed', 42),
        norm_params_path=str(checkpoints_dir / 'norm_params.json'))

    with open(output_dir / 'config.yaml', 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    arch = config['model']['architecture']
    c_in, c_out, nf = config['model']['c_in'], config['model']['c_out'], config['model']['nf']

    if arch == 'ResNet':
        model = create_model('ResNet', c_in, c_out).to(device)
        best_f1 = train_single_model(
            model, train_dataset, val_dataset, config, device,
            str(checkpoints_dir / 'resnet_baseline.pt'),
            str(logs_dir / 'resnet_training.csv'), 'ResNet')
        logger.info(f"ResNet: best val F1 = {best_f1:.4f}")
    else:
        ensemble_size = config['model'].get('ensemble_size', 5)
        ensemble_seeds = config['model'].get('ensemble_seeds', list(range(ensemble_size)))
        f1s = []
        for i, mseed in enumerate(ensemble_seeds):
            logger.info(f"\n{'='*60}")
            logger.info(f"Training member {i+1}/{ensemble_size} (seed={mseed})")
            torch.manual_seed(mseed)
            np.random.seed(mseed)
            model = InceptionTimeNetwork(c_in, c_out, nf).to(device)
            best_f1 = train_single_model(
                model, train_dataset, val_dataset, config, device,
                str(checkpoints_dir / f'inception_{i}_best.pt'),
                str(logs_dir / f'inception_{i}_training.csv'),
                f'Inception-{i}')
            f1s.append(best_f1)
        logger.info(f"\nEnsemble complete. F1s: {[f'{f:.4f}' for f in f1s]}, mean: {np.mean(f1s):.4f}")


if __name__ == '__main__':
    main()

### config.yaml

In [ ]:
%%writefile config.yaml
# Phase 1: Synthetic Pretraining Configuration
# =============================================
# Round 2: FocalLoss + class-specific alpha to fix poorly performing classes

experiment:
  name: phase1_synthetic_v2
  seed: 42

data:
  dataset_dir: ../data/synthetic_v2
  window_size: 2000
  window_stride: 1000
  raw_channels: [torque_ftlbs, rpm, pressure_psi, oil_temp_f, turns, hookload_klbs]
  derived_channels: [d_torque_dt, d_torque_dturns, torque_norm, turns_norm, phase, mask]
  total_channels: 12
  split_ratio: [0.70, 0.15, 0.15]
  split_seed: 42
  num_workers: 0
  pin_memory: false
  fault_threshold: 0.20    # Increased from 0.10 — stricter per-window labeling

model:
  architecture: InceptionTime
  c_in: 12
  c_out: 9
  nf: 32
  depth: 6
  ensemble_size: 5
  ensemble_seeds: [0, 1, 2, 3, 4]

training:
  optimizer: AdamW
  lr: 0.001
  weight_decay: 0.0001
  warmup_epochs: 5
  scheduler_eta_min: 0.00001
  batch_size: 64
  max_epochs: 120            # More epochs to let focal loss converge
  early_stop_patience: 20    # More patience — focal loss is slower to converge
  early_stop_metric: val_macro_f1
  gradient_clip: 1.0
  mixed_precision: true

loss:
  # FocalLoss: gamma down-weights easy examples, forcing model to learn
  # the hard classes (galling, wrong_compound, misaligned_stab).
  type: FocalLoss
  gamma: 1.5
  label_smoothing: 0.05
  # Per-class alpha: heavily upweight poorly performing classes.
  # Based on confusion matrix analysis:
  #   normal(0.90) cross(0.52) galling(0.17) stripped(1.0) over(0.78)
  #   under(0.48)  wrong(0.13)  misaligned(0.25) stall(0.66)
  # Upweight galling, wrong_compound, misaligned most aggressively.
  alpha: [0.5, 1.0, 3.0, 1.0, 1.0, 1.0, 3.0, 2.5, 1.0]

augmentation:
  mixup_alpha: 0.2
  mixup_prob: 0.3            # Enable mixup — helps with confused classes

sampler:
  type: BalancedBatchSampler
  samples_per_class: 7

class_map:
  normal_casing_ltc: 0
  normal_casing_btc: 0
  normal_casing_premium: 0
  normal_drill_pipe: 0
  normal_tubing: 0
  normal_breakout: 0
  full_cycle: 0
  multi_connection: 0
  cold_start: 0
  hot_environment: 0
  cross_thread: 1
  connection_jump: 1
  stick_slip: 1
  staged_fault: 1
  galling: 2
  stripped_thread: 3
  over_torque: 4
  washout: 4
  under_torque: 5
  wrong_compound: 6
  misaligned_stabbing: 7
  stall: 8

class_names:
  - normal_makeup
  - cross_thread
  - galling
  - stripped_thread
  - over_torque
  - under_torque
  - wrong_compound
  - misaligned_stab
  - stall

In [ ]:
import os
modules = ['features.py','dataset.py','models.py','losses.py',
           'sampler.py','eval.py','train.py','config.yaml']
for m in modules:
    size = os.path.getsize(m)
    print(f"  {m:<20s}  {size:>7,} bytes  OK")
print("\nAll modules written.")


## Step 7 — Prepare Datasets

Loads the manifest, splits scenarios into train/val/test, applies sliding windows
and Z-score normalization. Normalization parameters are computed from the training
split only (never leaked from val/test).

**If `manifest.csv` is missing**, the pipeline scans all sensor files, reads the
`fault_code` column from each, and deterministically classifies scenarios using the
physics engine’s FaultCode bitmask. The auto-built manifest is saved for future runs.

In [ ]:
import numpy as np
import torch
import torch.backends.cudnn as cudnn
import yaml
import os
from pathlib import Path
from torch.utils.data import DataLoader

# ── GPU Setup ─────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    free_mem, total_mem = torch.cuda.mem_get_info(0)
    print(f"GPU: {gpu_name} | {total_mem // 1024**2} MB | Free: {free_mem // 1024**2} MB")
    cudnn.benchmark = True
    cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    cudnn.allow_tf32 = True
    torch.cuda.empty_cache()
else:
    print("No GPU detected — falling back to CPU")

# ── Config ────────────────────────────────────────────────────────
with open('config.yaml') as f:
    config = yaml.safe_load(f)

config['data']['dataset_dir'] = DRIVE_DATA_DIR
config['model']['architecture'] = ARCHITECTURE
config['training']['max_epochs'] = MAX_EPOCHS
config['training']['batch_size'] = BATCH_SIZE
config['training']['lr'] = LEARNING_RATE
config['model']['ensemble_size'] = ENSEMBLE_SIZE
if SMOKE_TEST:
    config['training']['max_epochs'] = 5
    config['training']['early_stop_patience'] = 3

# ── Seed ──────────────────────────────────────────────────────────
seed = config['experiment']['seed']
torch.manual_seed(seed)
np.random.seed(seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(seed)

# ── Purge stale manifest ─────────────────────────────────────────
old_manifest = os.path.join(DRIVE_DATA_DIR, 'manifest.csv')
if os.path.exists(old_manifest):
    os.remove(old_manifest)
    print(f"Deleted old manifest (will rebuild with per-window labeling)")

# ── Prepare Datasets ─────────────────────────────────────────────
from dataset import prepare_datasets

checkpoints_dir = os.path.join(LOCAL_RESULTS_DIR, 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)

print("Loading and windowing dataset (with per-window fault labeling)...")
fault_threshold = config['data'].get('fault_threshold', 0.10)
train_dataset, val_dataset, test_dataset, norm_params = prepare_datasets(
    dataset_dir=DRIVE_DATA_DIR,
    class_map=config['class_map'],
    window_size=config['data']['window_size'],
    stride=config['data']['window_stride'],
    split_ratio=tuple(config['data']['split_ratio']),
    split_seed=config['data'].get('split_seed', 42),
    norm_params_path=os.path.join(checkpoints_dir, 'norm_params.json'),
    fault_threshold=fault_threshold,
)

# ── Data Quality Checks ──────────────────────────────────────────
print(f"\n--- Data Quality Checks ---")
print(f"Fault threshold: {fault_threshold} (windows need >= {fault_threshold*100:.0f}% fault activity)")

# Check a few samples
sample_x, sample_y = train_dataset[0]
print(f"Sample shape: {sample_x.shape} (expected [12, 2000])")
print(f"Sample dtype: {sample_x.dtype}")
print(f"Sample range: [{sample_x.min():.2f}, {sample_x.max():.2f}]")

# Per-channel stats
for ch in range(min(12, sample_x.shape[0])):
    ch_data = sample_x[ch]
    print(f"  Ch {ch:2d}: mean={ch_data.mean():.3f}, std={ch_data.std():.3f}, "
          f"min={ch_data.min():.3f}, max={ch_data.max():.3f}")

# ── DataLoader Setup ──────────────────────────────────────────────
NUM_WORKERS = min(4, os.cpu_count() or 1)
PREFETCH = 2
PIN = (device.type == 'cuda')

print(f"\nDataLoader: {NUM_WORKERS} workers | pin_memory={PIN} | batch={config['training']['batch_size']}")
print(f"Device: {device} | cuDNN benchmark: {cudnn.benchmark}")
print(f"Loss type: {config['loss'].get('type', 'FocalLoss')}")
print(f"Mixup prob: {config['augmentation'].get('mixup_prob', 0.0)}")
print(f"Warmup epochs: {config['training'].get('warmup_epochs', 0)}")

## Step 8 — Train

Trains the selected architecture.

- **ResNet**: ~30 min on T4 GPU. Good baseline (target >80% macro F1).
- **InceptionTime ensemble**: ~3–6 hrs on T4 GPU (5 members). Target >85% macro F1.

Checkpoints are saved after each epoch improvement.
Training auto-stops if validation F1 does not improve for 15 epochs.


In [ ]:
import os, torch, numpy as np
import torch.nn as nn
from pathlib import Path

from models import create_model, InceptionTimeNetwork, ResNetBaseline, count_parameters
from train import train_single_model

if 'DEVICE' not in dir():
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'LOCAL_RESULTS_DIR' not in dir():
    LOCAL_RESULTS_DIR = '/content/results'
if 'checkpoints_dir' not in dir():
    checkpoints_dir = os.path.join(LOCAL_RESULTS_DIR, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)

device = torch.device(DEVICE)
print(f"Training on: {device}")

arch = config['model']['architecture']
c_in = config['model']['c_in']
c_out = config['model']['c_out']
nf   = config['model']['nf']

logs_dir = os.path.join(LOCAL_RESULTS_DIR, 'logs')
os.makedirs(logs_dir, exist_ok=True)

# ── Step 1: ResNet Sanity Check (quick, ~5 min) ──────────────────
print(f"\n{'='*60}")
print("ResNet Sanity Check (20 epochs) — validates data pipeline")
print(f"{'='*60}")

sanity_config = {**config}
sanity_config = dict(config)  # shallow copy
sanity_config['training'] = dict(config['training'])
sanity_config['training']['max_epochs'] = 20
sanity_config['training']['early_stop_patience'] = 10
sanity_config['training']['warmup_epochs'] = 3

resnet_model = ResNetBaseline(c_in, c_out).to(device)
print(f"ResNet parameters: {count_parameters(resnet_model):,}")

resnet_f1 = train_single_model(
    resnet_model, train_dataset, val_dataset, sanity_config, device,
    checkpoint_path=os.path.join(checkpoints_dir, 'resnet_sanity.pt'),
    log_path=os.path.join(logs_dir, 'resnet_sanity.csv'),
    model_name='ResNet-Sanity',
)
print(f"\nResNet sanity check F1: {resnet_f1:.4f}")

if resnet_f1 < 0.30:
    print("\n*** WARNING: ResNet F1 < 0.30 — likely data pipeline issue! ***")
    print("Check: class distribution, NaN values, label correctness")
    print("Proceeding with InceptionTime anyway, but results may be poor.")
elif resnet_f1 < 0.50:
    print(f"\nResNet F1 = {resnet_f1:.4f} — moderate. Pipeline likely OK.")
    print("InceptionTime ensemble should improve this significantly.")
else:
    print(f"\nResNet F1 = {resnet_f1:.4f} — good! Pipeline validated.")

# ── Step 2: Train main architecture ──────────────────────────────
if arch == 'ResNet':
    print(f"\n{'='*60}")
    print("Training ResNet (full run)...")
    print(f"{'='*60}")
    model = ResNetBaseline(c_in, c_out).to(device)
    print(f"Parameters: {count_parameters(model):,}")
    best_f1 = train_single_model(
        model, train_dataset, val_dataset, config, device,
        checkpoint_path=os.path.join(checkpoints_dir, 'resnet_baseline.pt'),
        log_path=os.path.join(logs_dir, 'resnet_training.csv'),
        model_name='ResNet',
    )
    print(f"\nResNet best val F1: {best_f1:.4f}")

else:
    ensemble_size = config['model'].get('ensemble_size', 5)
    ensemble_seeds = config['model'].get('ensemble_seeds', list(range(ensemble_size)))
    f1_scores = []

    for i, member_seed in enumerate(ensemble_seeds):
        print(f"\n{'='*60}")
        print(f"Training InceptionTime member {i+1}/{ensemble_size}  (seed={member_seed})")

        torch.manual_seed(member_seed)
        np.random.seed(member_seed)
        model = InceptionTimeNetwork(c_in, c_out, nf).to(device)
        print(f"Parameters: {count_parameters(model):,}")

        best_f1 = train_single_model(
            model, train_dataset, val_dataset, config, device,
            checkpoint_path=os.path.join(checkpoints_dir, f'inception_{i}_best.pt'),
            log_path=os.path.join(logs_dir, f'inception_{i}_training.csv'),
            model_name=f'Inception-{i}',
        )
        f1_scores.append(best_f1)
        print(f"  Member {i} best F1: {best_f1:.4f}")

    print(f"\n{'='*60}")
    print(f"Ensemble training complete.")
    print(f"Individual F1s : {[f'{f:.4f}' for f in f1_scores]}")
    print(f"Mean F1        : {np.mean(f1_scores):.4f}")

## Step 9 — Evaluate on Test Set

Loads the best checkpoints and runs evaluation on the held-out test set.


In [ ]:
import os, torch, yaml
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader

from models import InceptionTimeNetwork, ResNetBaseline
from eval import (
    load_ensemble, ensemble_predict, compute_confusion_matrix,
    compute_per_class_metrics, assess_exit_criteria,
    save_confusion_matrix_csv, save_confusion_matrix_text, generate_report,
)

if 'DEVICE' not in dir():
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'LOCAL_RESULTS_DIR' not in dir():
    LOCAL_RESULTS_DIR = '/content/results'
if 'checkpoints_dir' not in dir():
    checkpoints_dir = os.path.join(LOCAL_RESULTS_DIR, 'checkpoints')
if 'config' not in dir():
    with open('config.yaml') as _f:
        config = yaml.safe_load(_f)

device = torch.device(DEVICE)
arch = config['model']['architecture']
c_in = config['model']['c_in']
c_out = config['model']['c_out']
nf   = config['model']['nf']
class_names = config['class_names']
num_classes = len(class_names)  # = c_out = 9

test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

if arch == 'ResNet':
    from pathlib import Path
    ckpt_path = os.path.join(checkpoints_dir, 'resnet_baseline.pt')
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = ResNetBaseline(c_in, c_out)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device).eval()
    models = [model]
    model_label = "ResNet baseline"
else:
    ensemble_size = config['model'].get('ensemble_size', 5)
    models = load_ensemble(Path(checkpoints_dir), device, c_in, c_out, nf, ensemble_size)
    model_label = f"InceptionTime ensemble ({len(models)} members)"

print(f"Loaded {len(models)} model(s): {model_label}")

# Inference
preds, labels, probs = ensemble_predict(models, test_loader, device)

# Metrics — using correct num_classes
cm = compute_confusion_matrix(preds, labels, num_classes)
per_class = compute_per_class_metrics(preds, labels, num_classes)
macro_f1 = np.mean([m['f1'] for m in per_class])
criteria = assess_exit_criteria(per_class, class_names)

# Print
print(f"\n{'='*60}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"\nPer-class metrics:")
print(f"{'Class':>20s} {'P':>8s} {'R':>8s} {'F1':>8s} {'N':>8s}")
for m, name in zip(per_class, class_names):
    print(f"{name:>20s} {m['precision']:>8.4f} {m['recall']:>8.4f} "
          f"{m['f1']:>8.4f} {m['support']:>8d}")

print(f"\nConfusion Matrix:")
print(save_confusion_matrix_text(cm, class_names))

print(f"\nExit Criteria:")
for c in criteria[:5]:
    marker = '[PASS]' if c['status'] == 'PASS' else '[FAIL]'
    print(f"  {marker} {c['criterion']}: {c['value']}  (target: {c['target']})")

# Save report
from pathlib import Path
generate_report(
    Path(LOCAL_RESULTS_DIR), class_names, macro_f1, per_class, criteria,
    cm, len(models), model_label=model_label
)
save_confusion_matrix_csv(cm, class_names,
    os.path.join(LOCAL_RESULTS_DIR, 'confusion_matrix.csv'))
print(f"\nReport saved to {LOCAL_RESULTS_DIR}/phase1_report.md")

## Step 10 — Save Results to Google Drive

Copies checkpoints, logs, and evaluation reports to Drive.


In [ ]:
import shutil, os

# Ensure globals from earlier cells are available
if 'LOCAL_RESULTS_DIR' not in dir():
    LOCAL_RESULTS_DIR = '/content/results'
if 'DRIVE_RESULTS_DIR' not in dir():
    DRIVE_RESULTS_DIR = '/content/drive/MyDrive/topdrive_ai/results'

print(f"Copying results to Drive...")
shutil.copytree(LOCAL_RESULTS_DIR, DRIVE_RESULTS_DIR, dirs_exist_ok=True)
print(f"Done! Results saved to: {DRIVE_RESULTS_DIR}")

# List checkpoints
ckpt_drive = os.path.join(DRIVE_RESULTS_DIR, 'checkpoints')
if os.path.exists(ckpt_drive):
    print("
Checkpoints:")
    for f in sorted(os.listdir(ckpt_drive)):
        path = os.path.join(ckpt_drive, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {f}  ({size_mb:.1f} MB)")

## Step 11 — Plot Training Curves (Optional)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

logs_dir = os.path.join(LOCAL_RESULTS_DIR, 'logs')
log_files = sorted([f for f in os.listdir(logs_dir) if f.endswith('.csv')])

if not log_files:
    print("No training logs found yet.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.tab10.colors

    for i, logfile in enumerate(log_files):
        df = pd.read_csv(os.path.join(logs_dir, logfile))
        label = logfile.replace('_training.csv', '')
        color = colors[i % len(colors)]
        axes[0].plot(df['epoch'], df['train_loss'],
                     label=f'{label} loss', color=color, alpha=0.7)
        axes[1].plot(df['epoch'], df['val_macro_f1'],
                     label=f'{label} F1', color=color)

    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Focal Loss')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title('Validation Macro F1')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Macro F1')
    axes[1].axhline(y=0.85, color='green', linestyle='--', label='Target 0.85')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(LOCAL_RESULTS_DIR, 'training_curves.png'), dpi=150)
    plt.show()
    print("Training curves saved.")
